# LoRA Trainer · Colab

UI-оболочка над kohya-ss/sd-scripts с поддержкой LoRA, LyCORIS и LoKr для NoobAI / Illustrious / Anima.

1. **Cell 1** — GPU runtime и Drive.
2. **Cell 2** — пути к датасету / моделям / output / samples.
3. **Cell 3** — установка зависимостей (~3 мин при первом запуске).
4. **Cell 4** — локальный кеш на SSD Colab + автосинк в Drive (крайне рекомендуется).
5. **Cell 5** — запуск сервера и cloudflared-туннеля.

В терминальном выводе появится `https://<random>.trycloudflare.com` — открывайте его в браузере.

## 1. GPU и Google Drive

In [ ]:
import subprocess, os
print(subprocess.check_output(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv']).decode())

from google.colab import drive
drive.mount('/content/drive')

## 2. Пути

Заполните поля справа. Остальные настройки — прямо в UI после запуска.

In [ ]:
import os

# @title Пути к папкам { display-mode: "form" }
DATASET_ROOT    = "/content/drive/MyDrive/lora/datasets/my-concept" # @param {type:"string", placeholder:"Папка с изображениями (любое название подпапок)"}
BASE_MODEL_ROOT = "/content/drive/MyDrive/lora/models"              # @param {type:"string", placeholder:"Папка с базовыми моделями (.safetensors)"}
OUTPUT_ROOT     = "/content/drive/MyDrive/lora/output/my-concept"   # @param {type:"string", placeholder:"Куда сохранять чекпоинты LoRA"}
SAMPLES_ROOT    = ""                                                  # @param {type:"string", placeholder:"Папка для сэмплов (оставьте пустым = OUTPUT_ROOT/sample)"}

# --------------------------------------------------------------------------
os.makedirs(OUTPUT_ROOT, exist_ok=True)
if SAMPLES_ROOT:
    os.makedirs(SAMPLES_ROOT, exist_ok=True)

for label, p in [('dataset', DATASET_ROOT), ('models', BASE_MODEL_ROOT), ('output', OUTPUT_ROOT)]:
    exists = os.path.isdir(p)
    icon = '✅' if exists else '❌ MISSING'
    print(f'{icon}  {label:8s}  {p}')

if not os.path.isdir(DATASET_ROOT):
    print('\n⚠️  Папка датасета не найдена — создайте её на Drive и перезапустите ячейку.')
if not os.path.isdir(BASE_MODEL_ROOT):
    print('⚠️  Папка моделей не найдена — положите .safetensors и перезапустите.')

# expose to FastAPI process
os.environ['LT_DATASET_ROOT']    = DATASET_ROOT
os.environ['LT_BASE_MODEL_ROOT'] = BASE_MODEL_ROOT
os.environ['LT_OUTPUT_ROOT']     = OUTPUT_ROOT
os.environ['LT_SAMPLES_ROOT']    = SAMPLES_ROOT
os.environ['LT_SD_SCRIPTS_DIR']  = '/content/sd-scripts'

print('\nПути сохранены. Запускайте ячейки 3 и 4.')

## 3. Установка

In [ ]:
# --- kohya-ss/sd-scripts -------------------------------------------
![ -d /content/sd-scripts ] || git clone --depth=1 https://github.com/kohya-ss/sd-scripts.git /content/sd-scripts

# --- chimerra-lora-trainer UI --------------------------------------
UI_REPO = 'https://github.com/DualChimerra/chimerra-lora-trainer.git'
![ -d /content/chimerra-lora-trainer ] || git clone --depth=1 $UI_REPO /content/chimerra-lora-trainer

# --- python deps ----------------------------------------------------
# sd-scripts requirements.txt содержит строку "." → пакет sd-scripts ставится
# в editable-режиме, для этого pip должен запускаться из его директории
%cd /content/sd-scripts
!pip install -q -r requirements.txt
%cd /content

!pip install -q lycoris-lora bitsandbytes prodigyopt fastapi 'uvicorn[standard]' python-multipart
# sd-scripts deps that sometimes don't ship via its own requirements.txt on Py 3.12
!pip install -q voluptuous einops safetensors ftfy

# --- cloudflared ----------------------------------------------------
!wget -q -O /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x /usr/local/bin/cloudflared

print('\n✓ Установка завершена.')

## 4. Локальный кеш и синхронизация (опционально, но рекомендуется)

Google Drive отдаёт случайный I/O в десятки раз медленнее локального SSD.
Из-за этого DataLoader тупит на каждой эпохе, а превью сэмплов в UI грузится секундами.
Эта ячейка зеркалит датасет и модели в `/content/local`, переводит OUTPUT туда же
и поднимает фоновый `rsync` обратно на Drive — чекпоинты и сэмплы появляются на Drive
сами через каждые `AUTO_SYNC_SECONDS`.

После ячейки доступны:
- `push_to_drive()` — слить локальный output на Drive прямо сейчас
- `pull_from_drive()` — подтянуть с Drive в локальный output (после рестарта runtime)


In [ ]:
# @title 4. Локальный кеш и автосинк { display-mode: "form" }
USE_LOCAL_DATASET = True   # @param {type:"boolean"}
USE_LOCAL_MODELS  = True   # @param {type:"boolean"}
USE_LOCAL_OUTPUT  = True   # @param {type:"boolean"}
AUTO_SYNC_SECONDS = 60     # @param {type:"integer"}
MIRROR_ALL_MODELS = False  # @param {type:"boolean"}
PREWARM_OUTPUT_FROM_DRIVE = False  # @param {type:"boolean"}

# Если MIRROR_ALL_MODELS=False — кеширует только один файл, который выбран
# в UI как base_model (определяется после первого запуска UI).
# Для первого запуска оставьте True или скопируйте нужный .safetensors вручную.

# --------------------------------------------------------------------
import os, subprocess, threading, time, shutil

LOCAL_BASE = '/content/local'
os.makedirs(LOCAL_BASE, exist_ok=True)

_drive_dataset = DATASET_ROOT
_drive_models  = BASE_MODEL_ROOT
_drive_output  = OUTPUT_ROOT
_drive_samples = SAMPLES_ROOT

def _rsync(src, dst, *, quiet=False, delete=False):
    if not src or not os.path.exists(src):
        return
    os.makedirs(dst, exist_ok=True)
    cmd = ['rsync', '-a']
    if delete: cmd.append('--delete')
    cmd.append('--quiet' if quiet else '--info=progress2')
    cmd += [src.rstrip('/') + '/', dst.rstrip('/') + '/']
    return subprocess.run(cmd, check=False).returncode

if USE_LOCAL_DATASET and os.path.isdir(DATASET_ROOT):
    local_ds = f'{LOCAL_BASE}/dataset'
    print(f'→ зеркалю датасет в {local_ds} …')
    _rsync(DATASET_ROOT, local_ds)
    DATASET_ROOT = local_ds
    os.environ['LT_DATASET_ROOT'] = DATASET_ROOT
    print('  ✓ готово')

if USE_LOCAL_MODELS and os.path.isdir(BASE_MODEL_ROOT):
    local_m = f'{LOCAL_BASE}/models'
    os.makedirs(local_m, exist_ok=True)
    if MIRROR_ALL_MODELS:
        print(f'→ зеркалю ВСЕ модели в {local_m} …')
        _rsync(BASE_MODEL_ROOT, local_m)
    else:
        # читаем какую модель выбрал UI; на первом запуске будет пусто
        wanted = None
        state_cfg = os.path.join(_drive_output, '.lora_trainer', 'last_config.json')
        if os.path.isfile(state_cfg):
            try:
                import json as _json
                wanted = _json.load(open(state_cfg)).get('paths', {}).get('base_model')
            except Exception: pass
        if wanted and os.path.isfile(wanted):
            dst = os.path.join(local_m, os.path.basename(wanted))
            if not os.path.isfile(dst) or os.path.getsize(dst) != os.path.getsize(wanted):
                print(f'→ копирую {os.path.basename(wanted)} в локальный кеш …')
                shutil.copy2(wanted, dst)
        else:
            print('  · пропускаю — base_model ещё не выбрана. Включи MIRROR_ALL_MODELS=True или скопируй файл вручную.')
    BASE_MODEL_ROOT = local_m
    os.environ['LT_BASE_MODEL_ROOT'] = BASE_MODEL_ROOT
    print('  ✓ готово')

_sync_lock = threading.Lock()

def push_to_drive(quiet=False):
    """Сбросить локальный output (чекпоинты + сэмплы) на Drive."""
    if not USE_LOCAL_OUTPUT:
        print('USE_LOCAL_OUTPUT=False — output уже на Drive')
        return
    with _sync_lock:
        _rsync(OUTPUT_ROOT, _drive_output, quiet=quiet)
    if not quiet:
        print(f'✓ {OUTPUT_ROOT} → {_drive_output}')

def pull_from_drive(quiet=False):
    """Подтянуть содержимое output с Drive обратно в локальный (после рестарта runtime)."""
    if not USE_LOCAL_OUTPUT:
        return
    with _sync_lock:
        _rsync(_drive_output, OUTPUT_ROOT, quiet=quiet)
    if not quiet:
        print(f'✓ {_drive_output} → {OUTPUT_ROOT}')

if USE_LOCAL_OUTPUT:
    local_out = f'{LOCAL_BASE}/output'
    os.makedirs(local_out, exist_ok=True)
    if PREWARM_OUTPUT_FROM_DRIVE and os.path.isdir(_drive_output):
        print(f'→ подтягиваю существующий output с Drive в {local_out} …')
        _rsync(_drive_output, local_out, quiet=True)
    elif os.path.isdir(_drive_output) and os.listdir(_drive_output):
        print(f'  · на Drive уже есть output ({_drive_output}) — НЕ копирую (PREWARM_OUTPUT_FROM_DRIVE=False).')
        print(f'    Чтобы догрузиться вручную: pull_from_drive()')
    OUTPUT_ROOT = local_out
    os.environ['LT_OUTPUT_ROOT'] = OUTPUT_ROOT
    os.environ['LT_DRIVE_OUTPUT_ROOT'] = _drive_output
    # state (last_config + presets) держим на Drive, чтобы пережить рестарт runtime
    os.environ['LT_STATE_DIR'] = os.path.join(_drive_output, '.lora_trainer')
    os.makedirs(os.environ['LT_STATE_DIR'], exist_ok=True)
    print(f'  ✓ output → {local_out}')
    print(f'  ✓ state  → {os.environ["LT_STATE_DIR"]} (на Drive)')

    if AUTO_SYNC_SECONDS > 0 and not globals().get('_sync_thread_started'):
        def _loop():
            while True:
                time.sleep(AUTO_SYNC_SECONDS)
                try: push_to_drive(quiet=True)
                except Exception as e: print(f'[sync] {e}')
        threading.Thread(target=_loop, daemon=True).start()
        globals()['_sync_thread_started'] = True
        print(f'  ✓ фоновая синхронизация в Drive каждые {AUTO_SYNC_SECONDS}s')

print()
print('Итоговые пути:')
print(f'  dataset : {DATASET_ROOT}')
print(f'  models  : {BASE_MODEL_ROOT}')
print(f'  output  : {OUTPUT_ROOT}')
print(f'  (drive output) : {_drive_output}')
print()
print('Команды в любой момент:')
print('  push_to_drive()   — сбросить output на Drive сейчас')
print('  pull_from_drive() — подтянуть output с Drive в локальный')


## 5. Запуск UI

Запускается FastAPI на :7860 и cloudflared-туннель. Ссылка появится ниже — открывайте её в браузере.

In [ ]:
import os, subprocess, time, re, signal, atexit, threading, sys

REPO_DIR = '/content/chimerra-lora-trainer'
PORT = 7860

# kill any previous instances
subprocess.run(['pkill', '-f', 'uvicorn'], check=False)
subprocess.run(['pkill', '-f', 'cloudflared'], check=False)
time.sleep(1)

env = os.environ.copy()
env['PYTHONPATH'] = REPO_DIR

server = subprocess.Popen(
    [sys.executable, '-m', 'uvicorn', 'backend.app:app', '--host', '0.0.0.0', '--port', str(PORT), '--log-level', 'warning'],
    cwd=REPO_DIR, env=env,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=1, text=True,
)

tunnel = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', f'http://127.0.0.1:{PORT}', '--no-autoupdate'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=1, text=True,
)

def _cleanup():
    for p in (server, tunnel):
        try: p.send_signal(signal.SIGINT); p.wait(timeout=5)
        except Exception:
            try: p.kill()
            except Exception: pass
atexit.register(_cleanup)

# --- wait for the trycloudflare URL ----------------------------------
tunnel_url = None
rx = re.compile(r'https://[a-z0-9\-]+\.trycloudflare\.com')
deadline = time.time() + 60
while time.time() < deadline and tunnel_url is None:
    line = tunnel.stdout.readline()
    if not line:
        time.sleep(0.2); continue
    sys.stdout.write(line)
    m = rx.search(line)
    if m: tunnel_url = m.group(0)

if tunnel_url:
    print('\n' + '═'*64)
    print(f'  UI:  {tunnel_url}')
    print('═'*64 + '\n')
else:
    print('Не удалось получить URL туннеля. Посмотрите логи выше.')

# stream server logs into the cell output in the background
def _stream(p, prefix):
    for line in p.stdout:
        sys.stdout.write(f'[{prefix}] {line}')
threading.Thread(target=_stream, args=(server, 'srv'), daemon=True).start()
threading.Thread(target=_stream, args=(tunnel, 'cf' ), daemon=True).start()

# keep this cell alive so the processes don't die
try:
    while True: time.sleep(60)
except KeyboardInterrupt:
    _cleanup()